# 08 — Trayectorias espectrales dentro del fruto: Δ respecto a día 0

## Proyecto
**Kaki_DeepHS_StorageDays**

## Objetivo científico

Eliminar explícitamente las diferencias espectrales basales entre frutos y preguntar:

> ¿Los cambios espectrales que ocurren **dentro del mismo fruto** contienen información suficiente para recuperar el tiempo de almacenamiento?

Para cada fruto \(i\) y día \(t>0\):

\[
\Delta R_{i,t}=R_{i,t}-R_{i,0}
\]

Se evalúan dos representaciones principales:

1. **delta_raw**
   \[
   R_{i,t}-R_{i,0}
   \]

2. **delta_snv**
   \[
   SNV(R_{i,t})-SNV(R_{i,0})
   \]

La segunda intenta reducir diferencias multiplicativas/aditivas y retener cambios de forma espectral.

## Por qué este control es importante

Si el modelo sigue prediciendo `storage_days` después de restar el espectro basal individual, la señal no puede atribuirse únicamente a diferencias estáticas entre frutos.

## Protocolos

### A. Within-sensor fruit-disjoint
VIS y VIS_COR se modelan por separado con los mismos folds congelados.

### B. Session-excluded
Para predecir el día \(d\), todas las observaciones de \(d\) se excluyen del training.

Se compara contra un `SessionExcluded_DummyMean` construido bajo la misma exclusión.

### C. Comparación con espectro absoluto
Se reutilizan las predicciones de los notebooks 02 y 03, restringidas a días \(>0\), para comparar:

\[
R_t
\quad \text{vs} \quad
R_t-R_0
\]

### D. Cross-sensor de trayectorias
Ambas cámaras se interpolan a una grilla común 410–900 nm.  
Se evalúa si las **trayectorias Δ**, no los niveles absolutos, pueden transferirse entre VIS y VIS_COR.

## Limitación esencial

Este análisis requiere disponer de una medición basal del mismo fruto en día 0. Por tanto, es un **control longitudinal/mecanístico**, no un modelo de despliegue de una sola medición.


### Public reproducibility version

This notebook is the cleaned repository version used to reproduce the manuscript analyses. The scientific workflow, frozen fruit-disjoint partitions, model-selection logic, seeds, and evaluation metrics are preserved. Google Drive mounting and chat-specific ZIP export steps were removed. Paths are relative to the repository root. Raw DeepHS Fruit data are not redistributed; see `../docs/DATASET.md`.


In [ ]:
from pathlib import Path
import os

def find_repository_root():
    """Locate the cloned repository or use PERSIMMON_HSI_ROOT if defined."""
    env_root = os.environ.get("PERSIMMON_HSI_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate

    if cwd.name == "notebooks":
        return cwd.parent

    return cwd

PROJECT_ROOT = find_repository_root()
print("Repository root:", PROJECT_ROOT)


## 1. Librerías, rutas y configuración


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import itertools
import json
import platform
import shutil
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_NAME = "Kaki_DeepHS_StorageDays"
NOTEBOOK_ID = "08_kaki_within_fruit_delta_trajectory"
RUN_SIGNATURE = "within_fruit_delta_v1"

ROOT = PROJECT_ROOT

INTERIM = ROOT / "data/interim"
SPLITS_DIR = ROOT / "data/splits"

BASE02 = ROOT / "results/02_kaki_fruit_disjoint_splits_baselines"
BASE03 = ROOT / "results/03_kaki_robustness_session_bias"

RESULTS = ROOT / "results/08_kaki_within_fruit_delta_trajectory"
FIGURES = ROOT / "figures/08_kaki_within_fruit_delta_trajectory"
TABLES = ROOT / "tables/08_kaki_within_fruit_delta_trajectory"
EXPORT_ROOT = ROOT / "exports/08_kaki_within_fruit_delta_trajectory"

for p in [
    RESULTS,
    FIGURES,
    TABLES,
    EXPORT_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

REPRESENTATIONS = [
    "delta_raw",
    "delta_snv",
]

RIDGE_ALPHAS = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
]

OUTER_FOLDS = 5
INNER_FOLDS = 4

COMMON_GRID = np.arange(
    410.0,
    900.0 + 0.001,
    5.0
)

RANDOM_SEED = 20260818
BOOTSTRAP_REPS = 5000

print(ROOT)
print("Representaciones:", REPRESENTATIONS)
print("Common grid:", len(COMMON_GRID), "puntos")


## 2. Cargar espectros, QC, splits y referencias


In [ ]:
SPECTRA_PATHS = {
    "VIS":
        INTERIM /
        "kaki_fruit_day_spectra_VIS.csv",

    "VIS_COR":
        INTERIM /
        "kaki_fruit_day_spectra_VIS_COR.csv",
}

CUBE_PATHS = {
    "VIS":
        INTERIM /
        "kaki_cube_spectra_VIS.csv",

    "VIS_COR":
        INTERIM /
        "kaki_cube_spectra_VIS_COR.csv",
}

OUTER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_outer5_fruit_disjoint_v1.csv"
)

INNER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_inner4_within_outer5_fruit_disjoint_v1.csv"
)

BASELINE_OOF_PATH = (
    BASE02 /
    "kaki_oof_predictions.csv"
)

SESSION_OOF_PATH = (
    BASE03 /
    "kaki_session_excluded_oof_predictions.csv"
)

for p in [
    *SPECTRA_PATHS.values(),
    *CUBE_PATHS.values(),
    OUTER_SPLIT_PATH,
    INNER_SPLIT_PATH,
    BASELINE_OOF_PATH,
    SESSION_OOF_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(
            f"Falta archivo requerido: {p}"
        )

expected_cube_counts = {
    "VIS": 395,
    "VIS_COR": 398,
}

spectra = {}
native_wavelengths = {}
qc_rows = []

for camera in [
    "VIS",
    "VIS_COR",
]:

    cube = pd.read_csv(
        CUBE_PATHS[
            camera
        ]
    )

    qc_rows.append({
        "camera_type":
            camera,

        "n_cubes":
            int(
                len(
                    cube
                )
            ),

        "median_mask_coverage":
            float(
                cube[
                    "mask_coverage"
                ].median()
            ),

        "n_mask_qc_not_ok":
            int(
                (
                    cube[
                        "mask_qc_flag"
                    ]
                    !=
                    "ok"
                ).sum()
            ),

        "n_extraction_errors":
            int(
                cube[
                    "extraction_error"
                ].notna().sum()
            ),
    })

    assert len(
        cube
    ) == expected_cube_counts[
        camera
    ]

    assert float(
        cube[
            "mask_coverage"
        ].median()
    ) < 0.87

    assert (
        cube[
            "mask_qc_flag"
        ]
        ==
        "ok"
    ).all()

    assert cube[
        "extraction_error"
    ].isna().all()

    df = pd.read_csv(
        SPECTRA_PATHS[
            camera
        ]
    )

    df[
        "fruit_id"
    ] = pd.to_numeric(
        df[
            "fruit_id"
        ],
        errors="raise"
    ).astype(int)

    df[
        "storage_days_model"
    ] = pd.to_numeric(
        df[
            "storage_days_model"
        ],
        errors="raise"
    ).astype(int)

    wl_cols = [
        c
        for c in df.columns
        if c.startswith(
            "wl_"
        )
    ]

    wl = np.array(
        [
            float(
                c.replace(
                    "wl_",
                    ""
                )
            )
            for c in wl_cols
        ],
        dtype=float
    )

    assert len(
        df
    ) == 199

    assert df[
        "fruit_id"
    ].nunique() == 40

    assert not df[
        wl_cols
    ].isna().any().any()

    spectra[
        camera
    ] = df.sort_values(
        [
            "fruit_id",
            "storage_days_model"
        ]
    ).reset_index(
        drop=True
    )

    native_wavelengths[
        camera
    ] = wl

qc = pd.DataFrame(
    qc_rows
)

display(
    qc
)

qc.to_csv(
    TABLES /
    "01_input_qc_gate.csv",
    index=False
)

outer = pd.read_csv(
    OUTER_SPLIT_PATH
)

inner = pd.read_csv(
    INNER_SPLIT_PATH
)

baseline_oof = pd.read_csv(
    BASELINE_OOF_PATH
)

session_oof_absolute = pd.read_csv(
    SESSION_OOF_PATH
)

for df in [
    outer,
    inner,
]:
    df[
        "fruit_id"
    ] = pd.to_numeric(
        df[
            "fruit_id"
        ],
        errors="raise"
    ).astype(int)

outer[
    "outer_fold"
] = pd.to_numeric(
    outer[
        "outer_fold"
    ],
    errors="raise"
).astype(int)

inner[
    "outer_fold"
] = pd.to_numeric(
    inner[
        "outer_fold"
    ],
    errors="raise"
).astype(int)

inner[
    "inner_fold"
] = pd.to_numeric(
    inner[
        "inner_fold"
    ],
    errors="raise"
).astype(int)

print("✅ Inputs cargados.")


## 3. Auditoría de splits


In [ ]:
all_fruits = set(
    outer[
        "fruit_id"
    ].astype(int)
)

audit_rows = []

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    train_fruits = (
        all_fruits
        -
        test_fruits
    )

    inner_fruits = set(
        inner.loc[
            inner[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    audit_rows.append({
        "outer_fold":
            int(
                outer_fold
            ),

        "n_train_fruits":
            len(
                train_fruits
            ),

        "n_test_fruits":
            len(
                test_fruits
            ),

        "train_test_overlap":
            len(
                train_fruits
                &
                test_fruits
            ),

        "inner_test_overlap":
            len(
                inner_fruits
                &
                test_fruits
            ),

        "inner_matches_train":
            inner_fruits
            ==
            train_fruits,
    })

split_audit = pd.DataFrame(
    audit_rows
)

display(
    split_audit
)

assert (
    split_audit[
        "train_test_overlap"
    ]
    ==
    0
).all()

assert (
    split_audit[
        "inner_test_overlap"
    ]
    ==
    0
).all()

assert split_audit[
    "inner_matches_train"
].all()

split_audit.to_csv(
    TABLES /
    "02_split_audit.csv",
    index=False
)

print("✅ Sin fuga de identidad.")


# 4. Construir representaciones Δ respecto al día 0

Para cada cámara y fruto:

\[
\Delta R_t=R_t-R_0
\]

Además:

\[
\Delta SNV_t = SNV(R_t)-SNV(R_0)
\]

Se excluye el día 0 de la tarea predictiva porque su vector Δ es exactamente cero.

El nuevo conjunto contiene:

\[
199-40=159
\]

observaciones fruto×día.


In [ ]:
def snv_rows(
    X
):

    X = np.asarray(
        X,
        dtype=float
    )

    mu = X.mean(
        axis=1,
        keepdims=True
    )

    sd = X.std(
        axis=1,
        keepdims=True
    )

    sd[
        (~np.isfinite(sd))
        |
        (sd == 0)
    ] = 1.0

    return (
        X
        -
        mu
    ) / sd


def build_delta_dataset(
    df
):

    spectral_columns = [
        c
        for c in df.columns
        if c.startswith(
            "wl_"
        )
    ]

    rows_raw = []
    rows_snv = []

    for fruit_id, g in df.groupby(
        "fruit_id"
    ):

        g = g.sort_values(
            "storage_days_model"
        )

        baseline = g[
            g[
                "storage_days_model"
            ]
            ==
            0
        ]

        assert len(
            baseline
        ) == 1

        X0 = baseline[
            spectral_columns
        ].to_numpy(
            dtype=float
        )

        X0_snv = snv_rows(
            X0
        )

        later = g[
            g[
                "storage_days_model"
            ]
            >
            0
        ].copy()

        X = later[
            spectral_columns
        ].to_numpy(
            dtype=float
        )

        X_snv = snv_rows(
            X
        )

        delta_raw = (
            X
            -
            X0
        )

        delta_snv = (
            X_snv
            -
            X0_snv
        )

        meta = later[
            [
                "fruit_id",
                "storage_days_model",
                "day_code",
            ]
        ].reset_index(
            drop=True
        )

        raw = meta.copy()

        snv = meta.copy()

        for j, col in enumerate(
            spectral_columns
        ):

            raw[
                col
            ] = delta_raw[
                :,
                j
            ]

            snv[
                col
            ] = delta_snv[
                :,
                j
            ]

        rows_raw.append(
            raw
        )

        rows_snv.append(
            snv
        )

    return {
        "delta_raw":
            pd.concat(
                rows_raw,
                ignore_index=True
            ),

        "delta_snv":
            pd.concat(
                rows_snv,
                ignore_index=True
            ),
    }


delta = {}

for camera in [
    "VIS",
    "VIS_COR",
]:

    delta[
        camera
    ] = build_delta_dataset(
        spectra[
            camera
        ]
    )

    for representation in REPRESENTATIONS:

        d = delta[
            camera
        ][
            representation
        ]

        assert len(
            d
        ) == 159

        assert d[
            "fruit_id"
        ].nunique() == 38

        assert set(
            d[
                "storage_days_model"
            ].unique()
        ) == {
            1,
            2,
            3,
            4,
            6,
            7,
            8,
        }

        print(
            camera,
            representation,
            d.shape
        )

print(
    "Nota: 38 frutos aportan días >0; "
    "2 frutos fueron destruidos en día 0."
)


## 5. Cobertura temporal del dataset Δ


In [ ]:
coverage = (
    delta[
        "VIS_COR"
    ][
        "delta_raw"
    ]
    .groupby(
        "storage_days_model"
    )
    .agg(
        n_fruit_days=(
            "fruit_id",
            "size"
        ),
        n_fruits=(
            "fruit_id",
            "nunique"
        ),
    )
    .reset_index()
)

display(
    coverage
)

coverage.to_csv(
    TABLES /
    "03_delta_temporal_coverage.csv",
    index=False
)


# 6. EDA de trayectorias Δ

Se grafica la curva media Δ por día para ambas representaciones y sensores.


In [ ]:
for camera in [
    "VIS",
    "VIS_COR",
]:

    wl = native_wavelengths[
        camera
    ]

    for representation in REPRESENTATIONS:

        d = delta[
            camera
        ][
            representation
        ]

        spectral_columns = [
            c
            for c in d.columns
            if c.startswith(
                "wl_"
            )
        ]

        fig, ax = plt.subplots(
            figsize=(9, 5)
        )

        for day, g in d.groupby(
            "storage_days_model"
        ):

            curve = g[
                spectral_columns
            ].mean().to_numpy(
                dtype=float
            )

            ax.plot(
                wl,
                curve,
                label=f"Día {int(day)}"
            )

        ax.axhline(
            0,
            linewidth=1
        )

        ax.set_xlabel(
            "Longitud de onda (nm)"
        )

        ax.set_ylabel(
            representation
        )

        ax.set_title(
            (
                f"{camera} — trayectoria media "
                f"{representation} respecto a día 0"
            )
        )

        ax.legend(
            ncol=2,
            fontsize=8
        )

        ax.grid(
            alpha=0.15
        )

        fig.tight_layout()

        fig.savefig(
            FIGURES /
            f"01_mean_{representation}_{camera}_by_day.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()


## 7. Magnitud longitudinal Δ por fruto y día


In [ ]:
magnitude_rows = []

for camera in [
    "VIS",
    "VIS_COR",
]:

    d = delta[
        camera
    ][
        "delta_raw"
    ]

    spectral_columns = [
        c
        for c in d.columns
        if c.startswith(
            "wl_"
        )
    ]

    X = d[
        spectral_columns
    ].to_numpy(
        dtype=float
    )

    rms = np.sqrt(
        np.mean(
            X ** 2,
            axis=1
        )
    )

    tmp = d[
        [
            "fruit_id",
            "storage_days_model",
        ]
    ].copy()

    tmp[
        "delta_rms"
    ] = rms

    tmp[
        "camera_type"
    ] = camera

    magnitude_rows.append(
        tmp
    )

delta_magnitude = pd.concat(
    magnitude_rows,
    ignore_index=True
)

magnitude_summary = (
    delta_magnitude.groupby(
        [
            "camera_type",
            "storage_days_model",
        ]
    )[
        "delta_rms"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std",
        ]
    )
    .reset_index()
)

display(
    magnitude_summary
)

magnitude_summary.to_csv(
    TABLES /
    "04_delta_magnitude_by_day.csv",
    index=False
)


# 8. Funciones de Ridge y métricas


In [ ]:
def fit_predict_ridge(
    X_train,
    y_train,
    X_test,
    alpha
):

    scaler = StandardScaler()

    X_train_s = scaler.fit_transform(
        X_train
    )

    X_test_s = scaler.transform(
        X_test
    )

    model = Ridge(
        alpha=float(
            alpha
        )
    )

    model.fit(
        X_train_s,
        y_train
    )

    return np.asarray(
        model.predict(
            X_test_s
        ),
        dtype=float
    )


def fruit_balanced_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return float(
        z.groupby(
            "fruit_id"
        )[
            "abs_error"
        ]
        .mean()
        .mean()
    )


def day_balanced_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return float(
        z.groupby(
            "y_true"
        )[
            "abs_error"
        ]
        .mean()
        .mean()
    )


def metric_bundle(
    df
):

    y = df[
        "y_true"
    ].to_numpy(
        dtype=float
    )

    p = df[
        "y_pred"
    ].to_numpy(
        dtype=float
    )

    residual = (
        p
        -
        y
    )

    rho, _ = spearmanr(
        y,
        p
    )

    return {
        "n_observations":
            int(
                len(
                    df
                )
            ),

        "n_fruits":
            int(
                df[
                    "fruit_id"
                ].nunique()
            ),

        "mae":
            float(
                mean_absolute_error(
                    y,
                    p
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    mean_squared_error(
                        y,
                        p
                    )
                )
            ),

        "r2":
            float(
                r2_score(
                    y,
                    p
                )
            ),

        "spearman_rho":
            float(
                rho
            ),

        "within_1_day_accuracy":
            float(
                np.mean(
                    np.abs(
                        residual
                    )
                    <=
                    1.0
                )
            ),

        "fruit_balanced_mae":
            fruit_balanced_mae(
                df
            ),

        "day_balanced_mae":
            day_balanced_mae(
                df
            ),

        "bias":
            float(
                np.mean(
                    residual
                )
            ),
    }


# 9. Nested CV estándar para Δ


In [ ]:
def inner_select_delta_standard(
    development_df,
    inner_assignment
):

    spectral_columns = [
        c
        for c in development_df.columns
        if c.startswith(
            "wl_"
        )
    ]

    best = None

    rows = []

    for alpha in RIDGE_ALPHAS:

        pred_parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            tr = development_df[
                ~development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            va = development_df[
                development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            # Algunos folds pueden incluir frutos sin observaciones >0.
            if len(
                va
            ) == 0:
                continue

            pred = fit_predict_ridge(
                tr[
                    spectral_columns
                ].to_numpy(
                    dtype=float
                ),
                tr[
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                va[
                    spectral_columns
                ].to_numpy(
                    dtype=float
                ),
                alpha
            )

            pred_parts.append(
                pd.DataFrame({
                    "fruit_id":
                        va[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        va[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        p = pd.concat(
            pred_parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            p
        )

        db = day_balanced_mae(
            p
        )

        mae = float(
            mean_absolute_error(
                p[
                    "y_true"
                ],
                p[
                    "y_pred"
                ]
            )
        )

        rows.append({
            "alpha":
                float(
                    alpha
                ),

            "inner_fruit_balanced_mae":
                fb,

            "inner_day_balanced_mae":
                db,

            "inner_mae":
                mae,
        })

        rank = (
            fb,
            db,
            mae,
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "alpha":
                    float(
                        alpha
                    ),

                "inner_fruit_balanced_mae":
                    fb,
            }

    return (
        best,
        pd.DataFrame(
            rows
        )
    )


# 10. OOF estándar: VIS y VIS_COR


In [ ]:
standard_parts = []
standard_selected_rows = []

for camera in [
    "VIS",
    "VIS_COR",
]:

    for representation in REPRESENTATIONS:

        d = delta[
            camera
        ][
            representation
        ]

        spectral_columns = [
            c
            for c in d.columns
            if c.startswith(
                "wl_"
            )
        ]

        for outer_fold in range(
            OUTER_FOLDS
        ):

            test_fruits = set(
                outer.loc[
                    outer[
                        "outer_fold"
                    ]
                    ==
                    outer_fold,
                    "fruit_id"
                ].astype(int)
            )

            dev = d[
                ~d[
                    "fruit_id"
                ].isin(
                    test_fruits
                )
            ]

            te = d[
                d[
                    "fruit_id"
                ].isin(
                    test_fruits
                )
            ]

            if len(
                te
            ) == 0:
                continue

            inner_assignment = inner[
                inner[
                    "outer_fold"
                ]
                ==
                outer_fold
            ][
                [
                    "fruit_id",
                    "inner_fold",
                ]
            ].copy()

            best, _ = inner_select_delta_standard(
                dev,
                inner_assignment
            )

            pred = fit_predict_ridge(
                dev[
                    spectral_columns
                ].to_numpy(
                    dtype=float
                ),
                dev[
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                te[
                    spectral_columns
                ].to_numpy(
                    dtype=float
                ),
                best[
                    "alpha"
                ]
            )

            standard_parts.append(
                pd.DataFrame({
                    "camera_type":
                        camera,

                    "representation":
                        representation,

                    "protocol":
                        "delta_standard",

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "alpha":
                        float(
                            best[
                                "alpha"
                            ]
                        ),

                    "fruit_id":
                        te[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "storage_days":
                        te[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        te[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

            standard_selected_rows.append({
                "camera_type":
                    camera,

                "representation":
                    representation,

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "alpha":
                    float(
                        best[
                            "alpha"
                        ]
                    ),

                "inner_fruit_balanced_mae":
                    float(
                        best[
                            "inner_fruit_balanced_mae"
                        ]
                    ),
            })

delta_standard_oof = pd.concat(
    standard_parts,
    ignore_index=True
)

delta_standard_selected = pd.DataFrame(
    standard_selected_rows
)

for camera in [
    "VIS",
    "VIS_COR",
]:

    for representation in REPRESENTATIONS:

        g = delta_standard_oof[
            (
                delta_standard_oof[
                    "camera_type"
                ]
                ==
                camera
            )
            &
            (
                delta_standard_oof[
                    "representation"
                ]
                ==
                representation
            )
        ]

        assert len(
            g
        ) == 159

        assert g[
            "fruit_id"
        ].nunique() == 38

print("✅ OOF estándar Δ completo.")


## 11. Métricas OOF estándar Δ


In [ ]:
standard_metric_rows = []

for (
    camera,
    representation
), g in delta_standard_oof.groupby(
    [
        "camera_type",
        "representation",
    ]
):

    m = metric_bundle(
        g
    )

    m.update({
        "camera_type":
            camera,

        "representation":
            representation,

        "protocol":
            "delta_standard",
    })

    standard_metric_rows.append(
        m
    )

delta_standard_metrics = pd.DataFrame(
    standard_metric_rows
).sort_values(
    [
        "camera_type",
        "fruit_balanced_mae",
    ]
)

display(
    delta_standard_metrics
)

delta_standard_metrics.to_csv(
    TABLES /
    "05_delta_standard_metrics.csv",
    index=False
)

delta_standard_selected.to_csv(
    TABLES /
    "06_delta_standard_selected_alpha.csv",
    index=False
)


# 12. Null estándar para días >0


In [ ]:
standard_null_parts = []

reference_delta_rows = delta[
    "VIS_COR"
][
    "delta_raw"
][
    [
        "fruit_id",
        "storage_days_model",
    ]
].copy()

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    tr = reference_delta_rows[
        ~reference_delta_rows[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ]

    te = reference_delta_rows[
        reference_delta_rows[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ]

    if len(
        te
    ) == 0:
        continue

    null_value = float(
        np.median(
            tr[
                "storage_days_model"
            ]
        )
    )

    standard_null_parts.append(
        pd.DataFrame({
            "outer_fold":
                int(
                    outer_fold
                ),

            "fruit_id":
                te[
                    "fruit_id"
                ].to_numpy(
                    dtype=int
                ),

            "storage_days":
                te[
                    "storage_days_model"
                ].to_numpy(
                    dtype=int
                ),

            "y_true":
                te[
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),

            "y_pred":
                np.repeat(
                    null_value,
                    len(
                        te
                    )
                ),
        })
    )

delta_standard_null = pd.concat(
    standard_null_parts,
    ignore_index=True
)

assert len(
    delta_standard_null
) == 159

delta_standard_null_metrics = metric_bundle(
    delta_standard_null
)

print(
    "Delta-task DummyMedian FB-MAE:",
    delta_standard_null_metrics[
        "fruit_balanced_mae"
    ]
)


# 13. Nested CV session-excluded para Δ


In [ ]:
def inner_select_delta_session_excluded(
    development_df,
    inner_assignment
):

    spectral_columns = [
        c
        for c in development_df.columns
        if c.startswith(
            "wl_"
        )
    ]

    best = None

    rows = []

    for alpha in RIDGE_ALPHAS:

        pred_parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            tr_all = development_df[
                ~development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            va_all = development_df[
                development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ]

            if len(
                va_all
            ) == 0:
                continue

            for target_day in sorted(
                va_all[
                    "storage_days_model"
                ].unique()
            ):

                tr = tr_all[
                    tr_all[
                        "storage_days_model"
                    ]
                    !=
                    target_day
                ]

                va = va_all[
                    va_all[
                        "storage_days_model"
                    ]
                    ==
                    target_day
                ]

                pred = fit_predict_ridge(
                    tr[
                        spectral_columns
                    ].to_numpy(
                        dtype=float
                    ),
                    tr[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),
                    va[
                        spectral_columns
                    ].to_numpy(
                        dtype=float
                    ),
                    alpha
                )

                pred_parts.append(
                    pd.DataFrame({
                        "fruit_id":
                            va[
                                "fruit_id"
                            ].to_numpy(
                                dtype=int
                            ),

                        "y_true":
                            va[
                                "storage_days_model"
                            ].to_numpy(
                                dtype=float
                            ),

                        "y_pred":
                            pred,
                    })
                )

        p = pd.concat(
            pred_parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            p
        )

        db = day_balanced_mae(
            p
        )

        mae = float(
            mean_absolute_error(
                p[
                    "y_true"
                ],
                p[
                    "y_pred"
                ]
            )
        )

        rows.append({
            "alpha":
                float(
                    alpha
                ),

            "inner_fruit_balanced_mae":
                fb,

            "inner_day_balanced_mae":
                db,

            "inner_mae":
                mae,
        })

        rank = (
            fb,
            db,
            mae,
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "alpha":
                    float(
                        alpha
                    ),

                "inner_fruit_balanced_mae":
                    fb,
            }

    return (
        best,
        pd.DataFrame(
            rows
        )
    )


# 14. OOF session-excluded Δ


In [ ]:
session_parts = []
session_selected_rows = []

for camera in [
    "VIS",
    "VIS_COR",
]:

    for representation in REPRESENTATIONS:

        d = delta[
            camera
        ][
            representation
        ]

        spectral_columns = [
            c
            for c in d.columns
            if c.startswith(
                "wl_"
            )
        ]

        for outer_fold in range(
            OUTER_FOLDS
        ):

            test_fruits = set(
                outer.loc[
                    outer[
                        "outer_fold"
                    ]
                    ==
                    outer_fold,
                    "fruit_id"
                ].astype(int)
            )

            dev = d[
                ~d[
                    "fruit_id"
                ].isin(
                    test_fruits
                )
            ]

            te_all = d[
                d[
                    "fruit_id"
                ].isin(
                    test_fruits
                )
            ]

            if len(
                te_all
            ) == 0:
                continue

            inner_assignment = inner[
                inner[
                    "outer_fold"
                ]
                ==
                outer_fold
            ][
                [
                    "fruit_id",
                    "inner_fold",
                ]
            ].copy()

            best, _ = (
                inner_select_delta_session_excluded(
                    dev,
                    inner_assignment
                )
            )

            for target_day in sorted(
                te_all[
                    "storage_days_model"
                ].unique()
            ):

                tr = dev[
                    dev[
                        "storage_days_model"
                    ]
                    !=
                    target_day
                ]

                te = te_all[
                    te_all[
                        "storage_days_model"
                    ]
                    ==
                    target_day
                ]

                pred = fit_predict_ridge(
                    tr[
                        spectral_columns
                    ].to_numpy(
                        dtype=float
                    ),
                    tr[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),
                    te[
                        spectral_columns
                    ].to_numpy(
                        dtype=float
                    ),
                    best[
                        "alpha"
                    ]
                )

                session_parts.append(
                    pd.DataFrame({
                        "camera_type":
                            camera,

                        "representation":
                            representation,

                        "protocol":
                            "delta_session_excluded",

                        "outer_fold":
                            int(
                                outer_fold
                            ),

                        "alpha":
                            float(
                                best[
                                    "alpha"
                                ]
                            ),

                        "fruit_id":
                            te[
                                "fruit_id"
                            ].to_numpy(
                                dtype=int
                            ),

                        "storage_days":
                            te[
                                "storage_days_model"
                            ].to_numpy(
                                dtype=int
                            ),

                        "y_true":
                            te[
                                "storage_days_model"
                            ].to_numpy(
                                dtype=float
                            ),

                        "y_pred":
                            pred,
                    })
                )

            session_selected_rows.append({
                "camera_type":
                    camera,

                "representation":
                    representation,

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "alpha":
                    float(
                        best[
                            "alpha"
                        ]
                    ),

                "inner_fruit_balanced_mae":
                    float(
                        best[
                            "inner_fruit_balanced_mae"
                        ]
                    ),
            })

delta_session_oof = pd.concat(
    session_parts,
    ignore_index=True
)

delta_session_selected = pd.DataFrame(
    session_selected_rows
)

for camera in [
    "VIS",
    "VIS_COR",
]:

    for representation in REPRESENTATIONS:

        g = delta_session_oof[
            (
                delta_session_oof[
                    "camera_type"
                ]
                ==
                camera
            )
            &
            (
                delta_session_oof[
                    "representation"
                ]
                ==
                representation
            )
        ]

        assert len(
            g
        ) == 159

print("✅ OOF session-excluded Δ completo.")


## 15. Métricas session-excluded Δ


In [ ]:
session_metric_rows = []

for (
    camera,
    representation
), g in delta_session_oof.groupby(
    [
        "camera_type",
        "representation",
    ]
):

    m = metric_bundle(
        g
    )

    m.update({
        "camera_type":
            camera,

        "representation":
            representation,

        "protocol":
            "delta_session_excluded",
    })

    session_metric_rows.append(
        m
    )

delta_session_metrics = pd.DataFrame(
    session_metric_rows
).sort_values(
    [
        "camera_type",
        "fruit_balanced_mae",
    ]
)

display(
    delta_session_metrics
)

delta_session_metrics.to_csv(
    TABLES /
    "07_delta_session_excluded_metrics.csv",
    index=False
)

delta_session_selected.to_csv(
    TABLES /
    "08_delta_session_excluded_selected_alpha.csv",
    index=False
)


# 16. Null correcto session-excluded para tarea Δ


In [ ]:
delta_session_null_parts = []

reference = delta[
    "VIS_COR"
][
    "delta_raw"
][
    [
        "fruit_id",
        "storage_days_model",
    ]
].copy()

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    dev = reference[
        ~reference[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ]

    te_all = reference[
        reference[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ]

    for target_day in sorted(
        te_all[
            "storage_days_model"
        ].unique()
    ):

        tr = dev[
            dev[
                "storage_days_model"
            ]
            !=
            target_day
        ]

        te = te_all[
            te_all[
                "storage_days_model"
            ]
            ==
            target_day
        ]

        null_value = float(
            tr[
                "storage_days_model"
            ].mean()
        )

        delta_session_null_parts.append(
            pd.DataFrame({
                "outer_fold":
                    int(
                        outer_fold
                    ),

                "fruit_id":
                    te[
                        "fruit_id"
                    ].to_numpy(
                        dtype=int
                    ),

                "storage_days":
                    te[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=int
                    ),

                "y_true":
                    te[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),

                "y_pred":
                    np.repeat(
                        null_value,
                        len(
                            te
                        )
                    ),
            })
        )

delta_session_null = pd.concat(
    delta_session_null_parts,
    ignore_index=True
)

assert len(
    delta_session_null
) == 159

delta_session_null_metrics = metric_bundle(
    delta_session_null
)

print(
    "Delta SessionExcluded_DummyMean FB-MAE:",
    delta_session_null_metrics[
        "fruit_balanced_mae"
    ]
)


# 17. Comparación con modelos de espectro absoluto

Para una comparación justa, las predicciones de los notebooks 02 y 03 se restringen a:

\[
storage\_days>0
\]

es decir, exactamente los 159 fruit-days de la tarea Δ.


In [ ]:
absolute_standard_rows = []
absolute_session_rows = []

for camera in [
    "VIS",
    "VIS_COR",
]:

    g = baseline_oof[
        (
            baseline_oof[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            baseline_oof[
                "model"
            ]
            ==
            "Ridge"
        )
        &
        (
            baseline_oof[
                "storage_days"
            ]
            >
            0
        )
    ].copy()

    assert len(
        g
    ) == 159

    m = metric_bundle(
        g
    )

    m.update({
        "camera_type":
            camera,

        "representation":
            "absolute_spectrum",

        "protocol":
            "standard_restricted_days_gt0",
    })

    absolute_standard_rows.append(
        m
    )

    h = session_oof_absolute[
        (
            session_oof_absolute[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            session_oof_absolute[
                "storage_days"
            ]
            >
            0
        )
    ].copy()

    assert len(
        h
    ) == 159

    m2 = metric_bundle(
        h
    )

    m2.update({
        "camera_type":
            camera,

        "representation":
            "absolute_spectrum",

        "protocol":
            "session_excluded_restricted_days_gt0",
    })

    absolute_session_rows.append(
        m2
    )

absolute_standard_metrics = pd.DataFrame(
    absolute_standard_rows
)

absolute_session_metrics = pd.DataFrame(
    absolute_session_rows
)

display(
    absolute_standard_metrics
)

display(
    absolute_session_metrics
)

absolute_standard_metrics.to_csv(
    TABLES /
    "09_absolute_standard_reference_days_gt0.csv",
    index=False
)

absolute_session_metrics.to_csv(
    TABLES /
    "10_absolute_session_reference_days_gt0.csv",
    index=False
)


# 18. Comparación maestra absolute vs Δ


In [ ]:
master_comparison = pd.concat(
    [
        absolute_standard_metrics,
        delta_standard_metrics,
        absolute_session_metrics,
        delta_session_metrics,
    ],
    ignore_index=True,
    sort=False
)

display(
    master_comparison.sort_values(
        [
            "camera_type",
            "protocol",
            "fruit_balanced_mae",
        ]
    )
)

master_comparison.to_csv(
    TABLES /
    "11_absolute_vs_delta_master_metrics.csv",
    index=False
)


# 19. Bootstrap pareado por fruto: Δ vs espectro absoluto y Δ vs null


In [ ]:
def per_fruit_mae(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return (
        z.groupby(
            "fruit_id"
        )[
            "abs_error"
        ]
        .mean()
    )


def paired_bootstrap(
    a,
    b,
    label,
    n_boot=5000,
    seed=20260818
):

    ids = sorted(
        set(
            a.index
        )
        &
        set(
            b.index
        )
    )

    av = a.loc[
        ids
    ].to_numpy(
        dtype=float
    )

    bv = b.loc[
        ids
    ].to_numpy(
        dtype=float
    )

    diff = (
        av
        -
        bv
    )

    rng = np.random.default_rng(
        seed
    )

    boot = np.empty(
        n_boot,
        dtype=float
    )

    for i in range(
        n_boot
    ):

        idx = rng.integers(
            0,
            len(
                diff
            ),
            size=len(
                diff
            )
        )

        boot[
            i
        ] = np.mean(
            diff[
                idx
            ]
        )

    return {
        "comparison":
            label,

        "n_fruits":
            int(
                len(
                    diff
                )
            ),

        "mean_delta_mae_days":
            float(
                diff.mean()
            ),

        "ci95_low":
            float(
                np.percentile(
                    boot,
                    2.5
                )
            ),

        "ci95_high":
            float(
                np.percentile(
                    boot,
                    97.5
                )
            ),

        "fraction_first_better":
            float(
                np.mean(
                    diff
                    <
                    0
                )
            ),
    }


bootstrap_rows = []

seed_offset = 0

standard_null_pf = per_fruit_mae(
    delta_standard_null
)

session_null_pf = per_fruit_mae(
    delta_session_null
)

for camera in [
    "VIS",
    "VIS_COR",
]:

    abs_std = baseline_oof[
        (
            baseline_oof[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            baseline_oof[
                "model"
            ]
            ==
            "Ridge"
        )
        &
        (
            baseline_oof[
                "storage_days"
            ]
            >
            0
        )
    ]

    abs_sess = session_oof_absolute[
        (
            session_oof_absolute[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            session_oof_absolute[
                "storage_days"
            ]
            >
            0
        )
    ]

    for representation in REPRESENTATIONS:

        dstd = delta_standard_oof[
            (
                delta_standard_oof[
                    "camera_type"
                ]
                ==
                camera
            )
            &
            (
                delta_standard_oof[
                    "representation"
                ]
                ==
                representation
            )
        ]

        dsess = delta_session_oof[
            (
                delta_session_oof[
                    "camera_type"
                ]
                ==
                camera
            )
            &
            (
                delta_session_oof[
                    "representation"
                ]
                ==
                representation
            )
        ]

        bootstrap_rows.append(
            paired_bootstrap(
                per_fruit_mae(
                    dstd
                ),
                per_fruit_mae(
                    abs_std
                ),
                (
                    f"{camera} {representation} standard "
                    "minus absolute standard"
                ),
                BOOTSTRAP_REPS,
                RANDOM_SEED + seed_offset
            )
        )

        seed_offset += 1

        bootstrap_rows.append(
            paired_bootstrap(
                per_fruit_mae(
                    dstd
                ),
                standard_null_pf,
                (
                    f"{camera} {representation} standard "
                    "minus delta DummyMedian"
                ),
                BOOTSTRAP_REPS,
                RANDOM_SEED + seed_offset
            )
        )

        seed_offset += 1

        bootstrap_rows.append(
            paired_bootstrap(
                per_fruit_mae(
                    dsess
                ),
                per_fruit_mae(
                    abs_sess
                ),
                (
                    f"{camera} {representation} session-excluded "
                    "minus absolute session-excluded"
                ),
                BOOTSTRAP_REPS,
                RANDOM_SEED + seed_offset
            )
        )

        seed_offset += 1

        bootstrap_rows.append(
            paired_bootstrap(
                per_fruit_mae(
                    dsess
                ),
                session_null_pf,
                (
                    f"{camera} {representation} session-excluded "
                    "minus correct delta null"
                ),
                BOOTSTRAP_REPS,
                RANDOM_SEED + seed_offset
            )
        )

        seed_offset += 1

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_results
)

bootstrap_results.to_csv(
    TABLES /
    "12_paired_fruit_bootstrap_delta_controls.csv",
    index=False
)


# 20. Error y bias por día


In [ ]:
day_rows = []

for protocol, pred_df in [
    (
        "delta_standard",
        delta_standard_oof
    ),
    (
        "delta_session_excluded",
        delta_session_oof
    ),
]:

    for (
        camera,
        representation,
        day
    ), g in pred_df.groupby(
        [
            "camera_type",
            "representation",
            "storage_days",
        ]
    ):

        err = (
            g[
                "y_pred"
            ].to_numpy(
                dtype=float
            )
            -
            g[
                "y_true"
            ].to_numpy(
                dtype=float
            )
        )

        day_rows.append({
            "protocol":
                protocol,

            "camera_type":
                camera,

            "representation":
                representation,

            "storage_days":
                int(
                    day
                ),

            "n":
                int(
                    len(
                        g
                    )
                ),

            "mae":
                float(
                    np.mean(
                        np.abs(
                            err
                        )
                    )
                ),

            "bias":
                float(
                    np.mean(
                        err
                    )
                ),
        })

day_metrics = pd.DataFrame(
    day_rows
)

display(
    day_metrics
)

day_metrics.to_csv(
    TABLES /
    "13_delta_error_bias_by_day.csv",
    index=False
)


# 21. Interpolar espectros absolutos a grilla común y construir Δ común


In [ ]:
def interpolate_to_common(
    df,
    native_grid
):

    spectral_columns = [
        c
        for c in df.columns
        if c.startswith(
            "wl_"
        )
    ]

    X = df[
        spectral_columns
    ].to_numpy(
        dtype=float
    )

    Y = np.empty(
        (
            len(
                df
            ),
            len(
                COMMON_GRID
            )
        ),
        dtype=float
    )

    for i in range(
        len(
            df
        )
    ):

        Y[
            i,
            :
        ] = np.interp(
            COMMON_GRID,
            native_grid,
            X[
                i,
                :
            ]
        )

    out = df[
        [
            "fruit_id",
            "storage_days_model",
            "day_code",
        ]
    ].copy()

    for j, wl in enumerate(
        COMMON_GRID
    ):

        out[
            f"cg_{wl:.1f}"
        ] = Y[
            :,
            j
        ]

    return out


common_absolute = {}

common_delta = {}

for camera in [
    "VIS",
    "VIS_COR",
]:

    ca = interpolate_to_common(
        spectra[
            camera
        ],
        native_wavelengths[
            camera
        ]
    )

    common_absolute[
        camera
    ] = ca

    cg_cols = [
        c
        for c in ca.columns
        if c.startswith(
            "cg_"
        )
    ]

    raw_parts = []
    snv_parts = []

    for fruit_id, g in ca.groupby(
        "fruit_id"
    ):

        g = g.sort_values(
            "storage_days_model"
        )

        b = g[
            g[
                "storage_days_model"
            ]
            ==
            0
        ]

        assert len(
            b
        ) == 1

        X0 = b[
            cg_cols
        ].to_numpy(
            dtype=float
        )

        X0_snv = snv_rows(
            X0
        )

        later = g[
            g[
                "storage_days_model"
            ]
            >
            0
        ].copy()

        X = later[
            cg_cols
        ].to_numpy(
            dtype=float
        )

        d_raw = (
            X
            -
            X0
        )

        d_snv = (
            snv_rows(
                X
            )
            -
            X0_snv
        )

        base_meta = later[
            [
                "fruit_id",
                "storage_days_model",
                "day_code",
            ]
        ].reset_index(
            drop=True
        )

        rr = base_meta.copy()

        ss = base_meta.copy()

        for j, c in enumerate(
            cg_cols
        ):

            rr[
                c
            ] = d_raw[
                :,
                j
            ]

            ss[
                c
            ] = d_snv[
                :,
                j
            ]

        raw_parts.append(
            rr
        )

        snv_parts.append(
            ss
        )

    common_delta[
        camera
    ] = {
        "delta_raw":
            pd.concat(
                raw_parts,
                ignore_index=True
            ),

        "delta_snv":
            pd.concat(
                snv_parts,
                ignore_index=True
            ),
    }

print("✅ Δ común 410–900 nm construido.")


# 22. Cross-sensor de trayectorias Δ

Aquí no usamos calibración afín: la pregunta es deliberadamente más estricta.

> ¿La trayectoria Δ de un sensor se puede aplicar directamente al otro sensor?

Se ajusta Ridge sobre el sensor fuente y se predice el sensor destino.


In [ ]:
def paired_common_delta(
    source_camera,
    target_camera,
    representation,
    fruit_ids,
    exclude_day=None
):

    src = common_delta[
        source_camera
    ][
        representation
    ]

    tgt = common_delta[
        target_camera
    ][
        representation
    ]

    src = src[
        src[
            "fruit_id"
        ].isin(
            fruit_ids
        )
    ].copy()

    tgt = tgt[
        tgt[
            "fruit_id"
        ].isin(
            fruit_ids
        )
    ].copy()

    if exclude_day is not None:

        src = src[
            src[
                "storage_days_model"
            ]
            !=
            exclude_day
        ]

        tgt = tgt[
            tgt[
                "storage_days_model"
            ]
            !=
            exclude_day
        ]

    src = src.sort_values(
        [
            "fruit_id",
            "storage_days_model"
        ]
    )

    tgt = tgt.sort_values(
        [
            "fruit_id",
            "storage_days_model"
        ]
    )

    assert np.array_equal(
        src[
            [
                "fruit_id",
                "storage_days_model",
            ]
        ].to_numpy(),
        tgt[
            [
                "fruit_id",
                "storage_days_model",
            ]
        ].to_numpy()
    )

    cg_cols = [
        c
        for c in src.columns
        if c.startswith(
            "cg_"
        )
    ]

    return {
        "source":
            src,

        "target":
            tgt,

        "features":
            cg_cols,
    }


def inner_select_cross_delta_standard(
    source_camera,
    target_camera,
    representation,
    development_fruits,
    inner_assignment
):

    best = None

    rows = []

    for alpha in RIDGE_ALPHAS:

        parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            train_fruits = (
                set(
                    development_fruits
                )
                -
                val_fruits
            )

            tr = paired_common_delta(
                source_camera,
                target_camera,
                representation,
                train_fruits
            )

            va = paired_common_delta(
                source_camera,
                target_camera,
                representation,
                val_fruits
            )

            if len(
                va[
                    "target"
                ]
            ) == 0:
                continue

            pred = fit_predict_ridge(
                tr[
                    "source"
                ][
                    tr[
                        "features"
                    ]
                ].to_numpy(
                    dtype=float
                ),
                tr[
                    "source"
                ][
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                va[
                    "target"
                ][
                    va[
                        "features"
                    ]
                ].to_numpy(
                    dtype=float
                ),
                alpha
            )

            parts.append(
                pd.DataFrame({
                    "fruit_id":
                        va[
                            "target"
                        ][
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        va[
                            "target"
                        ][
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        p = pd.concat(
            parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            p
        )

        db = day_balanced_mae(
            p
        )

        mae = float(
            mean_absolute_error(
                p[
                    "y_true"
                ],
                p[
                    "y_pred"
                ]
            )
        )

        rank = (
            fb,
            db,
            mae,
            float(
                alpha
            ),
        )

        rows.append({
            "alpha":
                float(
                    alpha
                ),

            "inner_fruit_balanced_mae":
                fb,

            "inner_day_balanced_mae":
                db,

            "inner_mae":
                mae,
        })

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "alpha":
                    float(
                        alpha
                    ),
            }

    return best


cross_directions = [
    (
        "VIS_COR",
        "VIS"
    ),
    (
        "VIS",
        "VIS_COR"
    ),
]

cross_delta_parts = []
cross_delta_selected_rows = []

for source_camera, target_camera in cross_directions:

    for representation in REPRESENTATIONS:

        for outer_fold in range(
            OUTER_FOLDS
        ):

            test_fruits = set(
                outer.loc[
                    outer[
                        "outer_fold"
                    ]
                    ==
                    outer_fold,
                    "fruit_id"
                ].astype(int)
            )

            development_fruits = (
                all_fruits
                -
                test_fruits
            )

            inner_assignment = inner[
                inner[
                    "outer_fold"
                ]
                ==
                outer_fold
            ][
                [
                    "fruit_id",
                    "inner_fold",
                ]
            ].copy()

            best = inner_select_cross_delta_standard(
                source_camera,
                target_camera,
                representation,
                development_fruits,
                inner_assignment
            )

            tr = paired_common_delta(
                source_camera,
                target_camera,
                representation,
                development_fruits
            )

            te = paired_common_delta(
                source_camera,
                target_camera,
                representation,
                test_fruits
            )

            if len(
                te[
                    "target"
                ]
            ) == 0:
                continue

            pred = fit_predict_ridge(
                tr[
                    "source"
                ][
                    tr[
                        "features"
                    ]
                ].to_numpy(
                    dtype=float
                ),
                tr[
                    "source"
                ][
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                te[
                    "target"
                ][
                    te[
                        "features"
                    ]
                ].to_numpy(
                    dtype=float
                ),
                best[
                    "alpha"
                ]
            )

            cross_delta_parts.append(
                pd.DataFrame({
                    "source_camera":
                        source_camera,

                    "target_camera":
                        target_camera,

                    "representation":
                        representation,

                    "protocol":
                        "cross_sensor_delta_direct",

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "alpha":
                        float(
                            best[
                                "alpha"
                            ]
                        ),

                    "fruit_id":
                        te[
                            "target"
                        ][
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "storage_days":
                        te[
                            "target"
                        ][
                            "storage_days_model"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        te[
                            "target"
                        ][
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

            cross_delta_selected_rows.append({
                "source_camera":
                    source_camera,

                "target_camera":
                    target_camera,

                "representation":
                    representation,

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "alpha":
                    float(
                        best[
                            "alpha"
                        ]
                    ),
            })

cross_delta_oof = pd.concat(
    cross_delta_parts,
    ignore_index=True
)

cross_delta_selected = pd.DataFrame(
    cross_delta_selected_rows
)

for source_camera, target_camera in cross_directions:

    for representation in REPRESENTATIONS:

        g = cross_delta_oof[
            (
                cross_delta_oof[
                    "source_camera"
                ]
                ==
                source_camera
            )
            &
            (
                cross_delta_oof[
                    "target_camera"
                ]
                ==
                target_camera
            )
            &
            (
                cross_delta_oof[
                    "representation"
                ]
                ==
                representation
            )
        ]

        assert len(
            g
        ) == 159

print("✅ Cross-sensor Δ directo completo.")


## 23. Métricas cross-sensor Δ


In [ ]:
cross_delta_metric_rows = []

for (
    source_camera,
    target_camera,
    representation
), g in cross_delta_oof.groupby(
    [
        "source_camera",
        "target_camera",
        "representation",
    ]
):

    m = metric_bundle(
        g
    )

    m.update({
        "source_camera":
            source_camera,

        "target_camera":
            target_camera,

        "representation":
            representation,

        "protocol":
            "cross_sensor_delta_direct",
    })

    cross_delta_metric_rows.append(
        m
    )

cross_delta_metrics = pd.DataFrame(
    cross_delta_metric_rows
).sort_values(
    "fruit_balanced_mae"
)

display(
    cross_delta_metrics
)

cross_delta_metrics.to_csv(
    TABLES /
    "14_cross_sensor_delta_metrics.csv",
    index=False
)

cross_delta_selected.to_csv(
    TABLES /
    "15_cross_sensor_delta_selected_alpha.csv",
    index=False
)


# 24. Figura principal — absolute vs Δ


In [ ]:
plot_rows = []

for camera in [
    "VIS",
    "VIS_COR",
]:

    abs_row = absolute_standard_metrics[
        absolute_standard_metrics[
            "camera_type"
        ]
        ==
        camera
    ].iloc[
        0
    ]

    plot_rows.append({
        "camera_type":
            camera,

        "representation":
            "absolute",

        "fruit_balanced_mae":
            float(
                abs_row[
                    "fruit_balanced_mae"
                ]
            ),
    })

    for representation in REPRESENTATIONS:

        r = delta_standard_metrics[
            (
                delta_standard_metrics[
                    "camera_type"
                ]
                ==
                camera
            )
            &
            (
                delta_standard_metrics[
                    "representation"
                ]
                ==
                representation
            )
        ].iloc[
            0
        ]

        plot_rows.append({
            "camera_type":
                camera,

            "representation":
                representation,

            "fruit_balanced_mae":
                float(
                    r[
                        "fruit_balanced_mae"
                    ]
                ),
        })

plot_df = pd.DataFrame(
    plot_rows
)

for camera in [
    "VIS",
    "VIS_COR",
]:

    g = plot_df[
        plot_df[
            "camera_type"
        ]
        ==
        camera
    ]

    fig, ax = plt.subplots(
        figsize=(7, 4.6)
    )

    ax.bar(
        g[
            "representation"
        ],
        g[
            "fruit_balanced_mae"
        ]
    )

    ax.axhline(
        delta_standard_null_metrics[
            "fruit_balanced_mae"
        ],
        linestyle="--",
        label="DummyMedian tarea Δ"
    )

    ax.set_ylabel(
        "MAE fruit-balanced (días)"
    )

    ax.set_title(
        f"{camera} — espectro absoluto vs trayectoria Δ"
    )

    ax.legend()

    ax.grid(
        axis="y",
        alpha=0.2
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"02_absolute_vs_delta_mae_{camera}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# 25. Figura — cross-sensor de trayectorias


In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 4.8)
)

labels = []

values = []

for _, row in cross_delta_metrics.iterrows():

    labels.append(
        (
            f"{row['source_camera']}→"
            f"{row['target_camera']}\n"
            f"{row['representation']}"
        )
    )

    values.append(
        float(
            row[
                "fruit_balanced_mae"
            ]
        )
    )

ax.bar(
    labels,
    values
)

ax.axhline(
    delta_standard_null_metrics[
        "fruit_balanced_mae"
    ],
    linestyle="--",
    label="DummyMedian tarea Δ"
)

ax.set_ylabel(
    "MAE fruit-balanced (días)"
)

ax.set_title(
    "Transferencia directa de trayectorias Δ"
)

ax.tick_params(
    axis="x",
    rotation=20
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.2
)

fig.tight_layout()

fig.savefig(
    FIGURES /
    "03_cross_sensor_delta_mae.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# 26. Guardar resultados centrales


In [ ]:
STANDARD_OOF_PATH = (
    RESULTS /
    "kaki_delta_standard_oof.csv"
)

SESSION_OOF_PATH_OUT = (
    RESULTS /
    "kaki_delta_session_excluded_oof.csv"
)

CROSS_DELTA_OOF_PATH = (
    RESULTS /
    "kaki_cross_sensor_delta_oof.csv"
)

STANDARD_NULL_PATH = (
    RESULTS /
    "kaki_delta_standard_dummy_median_oof.csv"
)

SESSION_NULL_PATH = (
    RESULTS /
    "kaki_delta_session_excluded_dummy_mean_oof.csv"
)

MASTER_METRICS_PATH = (
    RESULTS /
    "kaki_absolute_vs_delta_master_metrics.csv"
)

delta_standard_oof.to_csv(
    STANDARD_OOF_PATH,
    index=False
)

delta_session_oof.to_csv(
    SESSION_OOF_PATH_OUT,
    index=False
)

cross_delta_oof.to_csv(
    CROSS_DELTA_OOF_PATH,
    index=False
)

delta_standard_null.to_csv(
    STANDARD_NULL_PATH,
    index=False
)

delta_session_null.to_csv(
    SESSION_NULL_PATH,
    index=False
)

master_comparison.to_csv(
    MASTER_METRICS_PATH,
    index=False
)

print("✅ Resultados centrales guardados.")


# 27. Resumen científico automático


In [ ]:
def json_safe(
    obj
):

    if isinstance(
        obj,
        dict
    ):
        return {
            str(k):
                json_safe(
                    v
                )
            for k, v in obj.items()
        }

    if isinstance(
        obj,
        (
            list,
            tuple,
        )
    ):
        return [
            json_safe(
                v
            )
            for v in obj
        ]

    if isinstance(
        obj,
        np.integer
    ):
        return int(
            obj
        )

    if isinstance(
        obj,
        np.floating
    ):
        return (
            None
            if np.isnan(
                obj
            )
            else float(
                obj
            )
        )

    if isinstance(
        obj,
        np.bool_
    ):
        return bool(
            obj
        )

    if isinstance(
        obj,
        np.ndarray
    ):
        return json_safe(
            obj.tolist()
        )

    return obj


summary = {
    "project":
        PROJECT_NAME,

    "notebook":
        NOTEBOOK_ID,

    "run_signature":
        RUN_SIGNATURE,

    "timestamp":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "task": {
        "target":
            "storage_days_model",

        "days_included":
            [
                1,
                2,
                3,
                4,
                6,
                7,
                8,
            ],

        "n_fruit_days":
            159,

        "n_fruits_with_post_day0_observations":
            38,

        "baseline_required":
            True,
    },

    "representations":
        REPRESENTATIONS,

    "delta_standard_metrics":
        delta_standard_metrics.to_dict(
            orient="records"
        ),

    "delta_session_excluded_metrics":
        delta_session_metrics.to_dict(
            orient="records"
        ),

    "delta_standard_null":
        delta_standard_null_metrics,

    "delta_session_excluded_null":
        delta_session_null_metrics,

    "absolute_standard_reference_days_gt0":
        absolute_standard_metrics.to_dict(
            orient="records"
        ),

    "absolute_session_reference_days_gt0":
        absolute_session_metrics.to_dict(
            orient="records"
        ),

    "paired_bootstrap":
        bootstrap_results.to_dict(
            orient="records"
        ),

    "cross_sensor_delta":
        cross_delta_metrics.to_dict(
            orient="records"
        ),

    "limitations": [
        (
            "Delta models require a baseline day-0 measurement of the same fruit "
            "and therefore are longitudinal controls rather than single-scan "
            "deployment models."
        ),
        (
            "Only 38 fruits contribute observations after day 0 because two fruits "
            "were destructively sampled at baseline."
        ),
        (
            "Session-excluded evaluation still cannot causally separate biological "
            "time from acquisition session in this single campaign."
        ),
        (
            "Cross-sensor delta transfer remains within the same experimental campaign."
        ),
        (
            "External campaign validation is still required for deployment claims."
        ),
    ],
}

SUMMARY_PATH = (
    RESULTS /
    "kaki_within_fruit_delta_trajectory_summary.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    )
)


# 28. Notas automáticas de revisión


In [ ]:
review_lines = [
    "KAKI — WITHIN-FRUIT DELTA TRAJECTORY",
    "",
    "Task: predict storage_days > 0 after subtracting each fruit's day-0 spectrum.",
    "N = 159 fruit-days from 38 fruits.",
    "",
    (
        "Delta standard null FB-MAE = "
        f"{delta_standard_null_metrics['fruit_balanced_mae']:.4f} d"
    ),
    (
        "Delta session-excluded null FB-MAE = "
        f"{delta_session_null_metrics['fruit_balanced_mae']:.4f} d"
    ),
    "",
    "DELTA STANDARD:",
]

for _, r in delta_standard_metrics.iterrows():

    review_lines.append(
        (
            f"  {r['camera_type']} | {r['representation']}: "
            f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
            f"R2={r['r2']:.4f}, "
            f"±1d={r['within_1_day_accuracy']:.4f}"
        )
    )

review_lines.extend([
    "",
    "ABSOLUTE STANDARD REFERENCE, DAYS > 0:",
])

for _, r in absolute_standard_metrics.iterrows():

    review_lines.append(
        (
            f"  {r['camera_type']}: "
            f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
            f"R2={r['r2']:.4f}"
        )
    )

review_lines.extend([
    "",
    "DELTA SESSION-EXCLUDED:",
])

for _, r in delta_session_metrics.iterrows():

    review_lines.append(
        (
            f"  {r['camera_type']} | {r['representation']}: "
            f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
            f"R2={r['r2']:.4f}, "
            f"improvement-null="
            f"{delta_session_null_metrics['fruit_balanced_mae']-r['fruit_balanced_mae']:+.4f}"
        )
    )

review_lines.extend([
    "",
    "CROSS-SENSOR DELTA:",
])

for _, r in cross_delta_metrics.iterrows():

    review_lines.append(
        (
            f"  {r['source_camera']}->{r['target_camera']} | "
            f"{r['representation']}: "
            f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
            f"R2={r['r2']:.4f}"
        )
    )

review_lines.extend([
    "",
    "Important:",
    (
        "Delta models require the same fruit at day 0. "
        "This is a longitudinal mechanistic control, not a deployment model."
    ),
    (
        "If delta remains predictive, stable between-fruit baseline differences "
        "cannot fully explain the temporal signal."
    ),
    (
        "Review this package before deciding whether the classical-model phase is complete."
    ),
])

REVIEW_PATH = (
    RESULTS /
    "DELTA_TRAJECTORY_REVIEW_NOTES.txt"
)

REVIEW_PATH.write_text(
    "\n".join(
        review_lines
    )
    +
    "\n",
    encoding="utf-8"
)

print(
    REVIEW_PATH.read_text()
)
